In [ ]:
# now need to calibrate our rgb to wavelength conversion
# going to set calibration as helium bcos it has the widest range

helium_bands = { # pixel pos : real wavelength
    825: 388.8,
    1285: 447.1,
    1435: 471.3,
    1566: 492.1,
    1612: 501.5,
    2136: 587.5,
    2627: 667.8
}

#print(picFiles_list)
he_rgb = get_rgb_nobackground('pics/g6f_he_2_full.dng') # helium
display(he_rgb)

calibration = he_rgb[he_rgb["pixel"].isin(helium_bands) 
                     & he_rgb["valid"]].copy()
#print(calibration)
calibration["wavelength_nm"] = calibration["pixel"].map(helium_bands)

# gonna fit linear
# im pretty sure the camera response is NOT linear
# but we only have 6 data points from helium and i dont want to use the rest
# bcos iw to validate with them

X = calibration[["r", "g", "b"]].to_numpy()
y = calibration["wavelength_nm"].to_numpy()

model = Ridge(alpha = 0.001) 
# alpha does... something... to control regulation strength
# idk what but docs says it must be non negative number. okay
model.fit(X, y)
calibration["predicted_nm"] = model.predict(X)

display(calibration[["pixel", "wavelength_nm", "predicted_nm"]])


In [ ]:
# lets go see the rest

def test_element(pic, known_lines=None):
    # Extract RGB measurements using existing function
    data = get_rgb_nobackground(pic)

    # Exclude invalid RGB measurements
    data = data[data["valid"]].copy()

    # Predict wavelengths using the helium model
    X = data[["r", "g", "b"]].to_numpy()

    data["predicted_nm"] = model.predict(X)

    # put ur known lines
    if known_lines is not None:
        data["actual_nm"] = data["pixel"].map(known_lines)
        data["error_nm"] = data["predicted_nm"] - data["actual_nm"]

    # display(data[["pixel", "predicted_nm", "actual_nm", "error_nm"]].round(2))
    display(data)

    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(
        data["pixel"],
        data["predicted_nm"],
        s=60
    )

    for _, row in data.iterrows():
        ax.annotate(
            f'{row["predicted_nm"]:.1f}',
            (row["pixel"], row["predicted_nm"]),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=8
        )

    ax.set_xlabel("Detected peak position (pixels)")
    ax.set_ylabel("Predicted wavelength (nm)")
    ax.set_title(pic)
    ax.grid(alpha=0.3)

    plt.show()

    if known_lines is not None:
        matched = data.dropna(subset=["actual_nm"])

        if len(matched):
            mae = np.mean(np.abs(matched["error_nm"]))
            print(f"Validation MAE: {mae:.2f} nm")

    return data

In [ ]:
h_file = "pics/g6f_h_full.dng"

# pixel pos:legit
hydrogen_lines = {
    1732: 486,
    2666: 656 
}

h_results = test_element(
    h_file,
    known_lines=hydrogen_lines
)

In [ ]:
hg_file = "pics/g6f_hg_full.dng"

# pos:legit
mercury_lines = {
    1344: 404.7,
    1501: 435.8,
    2081: 546.1,
    2255: 576.9 # there's another at 579 which we can't resolve
}

hg_results = test_element(
    hg_file,
    known_lines=mercury_lines
)

In [ ]:
# this deuterium is highkey suspicious
# is it neon?????
# time to check

def unknown_ele(file):
    rgb = get_rgb_nobackground(file)

    valid = rgb[rgb["valid"]].copy()

    # Use the same features as your helium calibration
    sus = valid[["r", "g", "b"]].to_numpy()

    valid["estimated_nm"] = model.predict(sus)

    return valid

In [ ]:
sus_deu = unknown_ele("pics/g6f_deu_full.dng")

display(sus_deu[["pixel", "estimated_nm"]])

In [ ]:

datasets = {
    "Helium": (he, helium_bands),
    "Hydrogen": (h, hydrogen_lines),
    "Mercury": (hg, mercury_lines)
}

for name, (data, references) in datasets.items():

    test = data.copy()
    test["actual_nm"] = test["pixel"].map(references)

    test = test.dropna(subset=["actual_nm"])
    test = test[
        test["actual_nm"].between(400, 700)
    ].copy()

    test["original_nm"], _ = predict_sensor_wavelength(test)
    test["corrected_nm"], _ = predict_corrected_wavelength(test)

    original_mae = np.mean(
        np.abs(test["original_nm"] - test["actual_nm"])
    )

    corrected_mae = np.mean(
        np.abs(test["corrected_nm"] - test["actual_nm"])
    )

    print(f"\n{name}")
    display(test[
        ["pixel", "actual_nm", "original_nm", "corrected_nm"]
    ].round(2))

    print(f"Original MAE:  {original_mae:.2f} nm")
    print(f"Corrected MAE: {corrected_mae:.2f} nm")
